In [1]:
# ==============================================================================
# 🏆 DIABETES PREDICTION: ROBUST BASELINE (FIXED TYPES)
# ==============================================================================

# 📦 0. INSTALL MISSING LIBRARY
import os
os.system('pip install category_encoders')

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from category_encoders import TargetEncoder
import warnings

# Settings
warnings.filterwarnings('ignore')
SEED = 42
FOLDS = 5

# ==============================================================================
# 1. LOAD & SORT
# ==============================================================================
print("📥 Loading Data...")
train = pd.read_csv('/kaggle/input/playground-series-s5e12/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s5e12/test.csv')

# 🚨 CRITICAL: Sort by ID to align with the hidden time dimension
train = train.sort_values('id').reset_index(drop=True)

target = 'diagnosed_diabetes'
# We will define features later after engineering

print(f"✅ Data Loaded. Train: {train.shape}, Test: {test.shape}")

# ==============================================================================
# 2. ADVERSARIAL WEIGHTING (The "Smart" Weights)
# ==============================================================================
def get_weights(train, test):
    print("\n⚖️ Calculating Weights...")
    drift_cols = ['id', 'physical_activity_minutes_per_week', 'triglycerides', 
                  'cholesterol_total', 'ldl_cholesterol']
    
    # Label: 0 = Train, 1 = Test
    X = pd.concat([train[drift_cols], test[drift_cols]]).reset_index(drop=True)
    y = np.array([0]*len(train) + [1]*len(test))
    
    # Use CPU for small weight model
    clf = lgb.LGBMClassifier(n_estimators=50, max_depth=3, verbose=-1, random_state=SEED, device='cpu')
    
    preds = cross_val_predict(clf, X, y, cv=3, method='predict_proba')[:, 1]
    
    p_train = preds[:len(train)]
    p_train = np.clip(p_train, 0.01, 0.95)
    weights = p_train / (1 - p_train)
    weights = weights * (len(train) / weights.sum())
    
    print(f"   -> Max Weight: {weights.max():.2f} (Drift Detected!)")
    print(f"   -> Min Weight: {weights.min():.2f}")
    return weights

sample_weights = get_weights(train, test)

# ==============================================================================
# 3. ROBUST FEATURE ENGINEERING
# ==============================================================================
def engineer(df):
    df = df.copy()
    
    # A. Biological Interactions
    df['athero_index'] = np.log1p(df['triglycerides']) - np.log1p(df['hdl_cholesterol'])
    df['pulse_pressure'] = df['systolic_bp'] - df['diastolic_bp']
    df['risk_mult'] = df['family_history_diabetes'] * ((df['bmi']-25)/5)
    
    # B. Drift Handling (Quantile Binning)
    drift_vars = ['physical_activity_minutes_per_week', 'triglycerides', 
                  'cholesterol_total', 'ldl_cholesterol', 'age', 'bmi']
    
    for col in drift_vars:
        df[f'{col}_q10'] = pd.qcut(df[col], q=10, labels=False, duplicates='drop')
        df[f'{col}_log'] = np.log1p(df[col])
        
    return df

print("\n🛠️ Engineering Features...")
train_eng = engineer(train)
test_eng = engineer(test)

# ==============================================================================
# 3.5 FIX CATEGORICAL TYPES (CRITICAL FIX)
# ==============================================================================
print("🔄 Converting Object columns to Category...")
# Identify original categorical columns (strings)
cat_cols = train_eng.select_dtypes(include=['object']).columns.tolist()

for c in cat_cols:
    train_eng[c] = train_eng[c].astype('category')
    test_eng[c] = test_eng[c].astype('category')

print(f"   -> Converted: {cat_cols}")

# ==============================================================================
# 4. TRAINING LOOP (With Leak-Proof Encoding)
# ==============================================================================
te_cols = [c for c in train_eng.columns if '_q10' in c]
# Include ID, exclude target and raw TE cols (we encode them inside loop)
features = [c for c in train_eng.columns if c not in [target] + te_cols] 

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

print("\n🚀 Training LightGBM (GPU Enabled)...")

for fold, (idx_tr, idx_val) in enumerate(skf.split(train, train[target])):
    X_tr = train_eng.iloc[idx_tr].copy()
    X_val = train_eng.iloc[idx_val].copy()
    y_tr, y_val = train[target].iloc[idx_tr], train[target].iloc[idx_val]
    
    w_tr = sample_weights[idx_tr]
    
    # Target Encode INSIDE Fold
    enc = TargetEncoder(cols=te_cols, smoothing=20)
    X_tr[te_cols] = enc.fit_transform(X_tr[te_cols], y_tr)
    X_val[te_cols] = enc.transform(X_val[te_cols])
    
    # Prepare Test Fold
    X_test_fold = test_eng.copy()
    X_test_fold[te_cols] = enc.transform(X_test_fold[te_cols])
    
    # Train
    clf = lgb.LGBMClassifier(
        n_estimators=3000, 
        learning_rate=0.012, 
        num_leaves=64, 
        colsample_bytree=0.5, 
        subsample=0.8, 
        reg_lambda=1.5, 
        device='gpu',           # GPU ENABLED
        random_state=SEED, 
        verbose=-1
    )
    
    # Pass 'categorical_feature' automatically via pandas category types
    clf.fit(X_tr[features + te_cols], y_tr, sample_weight=w_tr, 
            eval_set=[(X_val[features + te_cols], y_val)], 
            eval_metric='auc',
            callbacks=[lgb.early_stopping(100, verbose=False)])
    
    # Predict
    oof_preds[idx_val] = clf.predict_proba(X_val[features + te_cols])[:,1]
    test_preds += clf.predict_proba(X_test_fold[features + te_cols])[:,1] / FOLDS
    
    print(f"   Fold {fold+1} AUC: {roc_auc_score(y_val, oof_preds[idx_val]):.5f}")

print(f"\n🏆 Overall CV: {roc_auc_score(train[target], oof_preds):.5f}")

# ==============================================================================
# 5. TAIL CHECK & SAVE
# ==============================================================================
tail_idx = int(len(train)*0.95)
tail_score = roc_auc_score(train[target].iloc[tail_idx:], oof_preds[tail_idx:])
print(f"🔍 Tail AUC (The Truth): {tail_score:.5f}")

pd.DataFrame({'id': test['id'], 'diagnosed_diabetes': test_preds}).to_csv('submission.csv', index=False)
print("✅ submission.csv saved successfully!")

📥 Loading Data...
✅ Data Loaded. Train: (700000, 26), Test: (300000, 25)

⚖️ Calculating Weights...
   -> Max Weight: 13.12 (Drift Detected!)
   -> Min Weight: 0.02

🛠️ Engineering Features...
🔄 Converting Object columns to Category...
   -> Converted: ['gender', 'ethnicity', 'education_level', 'income_level', 'smoking_status', 'employment_status']

🚀 Training LightGBM (GPU Enabled)...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


   Fold 1 AUC: 0.72249
   Fold 2 AUC: 0.72054
   Fold 3 AUC: 0.72085
   Fold 4 AUC: 0.72203
   Fold 5 AUC: 0.72178

🏆 Overall CV: 0.72154
🔍 Tail AUC (The Truth): 0.70485
✅ submission.csv saved successfully!
